# Previous Application Feature Engineering

Here we will turn previous application records into **one row for every applicant**.
`SK_ID_PREV` identifies an application, while `SK_ID_CURR` identifies the applicant.

هنجمع الطلبات السابقة لكل عميل، عشان يبقى عندنا صف واحد لكل `SK_ID_CURR`.

We follow the small-cell approach and English/Arabic explanations in `01_bureau.ipynb`.
The decisions below come from `03_previous_application_eda.ipynb`.
We use only `previous_application.csv`; the raw file stays unchanged.

## 1. Imports and Paths

We use the same `REPO_ROOT`, `DATA_RAW`, and `DATA_INTERIM` names as the Bureau notebook.
The path check lets us run from the project root or the notebook directory.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import hashlib
from IPython.display import display, Markdown

In [2]:
REPO_ROOT = Path.cwd().parent.parent.parent
DATA_RAW  = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
DATA_INTERIM= REPO_ROOT / "data" / "interim"
PREV_PATH = DATA_RAW / "previous_application.csv"
OUTPUT_PATH = DATA_INTERIM / "previous_application_features.parquet"

In [3]:
# A fingerprint will let us verify that the original CSV did not change.
with PREV_PATH.open("rb") as raw_file:
    raw_sha256_before = hashlib.file_digest(raw_file, "sha256").hexdigest()

## 2. Load the Dataset

The EDA found 1,670,214 rows and 37 source columns. We read the columns needed
for these features and the date audit. Repeated strings use `category` to save
memory; amount fields keep their original floating-point precision.

`XNA` and `XAP` remain explicit categories. Only empty CSV fields become missing.
We also read the header so the final report can distinguish the full source
dimensions from the selected columns loaded into memory.

In [4]:
source_columns = pd.read_csv(PREV_PATH, nrows=0).columns.tolist()
CATEGORY_COLUMNS = [
    "NAME_CONTRACT_TYPE", "NAME_CONTRACT_STATUS", "NAME_PAYMENT_TYPE",
    "NAME_GOODS_CATEGORY", "CHANNEL_TYPE", "PRODUCT_COMBINATION",
    "FLAG_LAST_APPL_PER_CONTRACT"
]
AMOUNT_COLUMNS = [
    "AMT_APPLICATION", "AMT_CREDIT", "AMT_ANNUITY",
    "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE"
]
DATE_COLUMNS = [
    "DAYS_DECISION", "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION"
]
USE_COLUMNS = ["SK_ID_PREV", "SK_ID_CURR", "CNT_PAYMENT"] + CATEGORY_COLUMNS + AMOUNT_COLUMNS + DATE_COLUMNS

prev = pd.read_csv(
    PREV_PATH,
    usecols=USE_COLUMNS,
    dtype={**{col: "category" for col in CATEGORY_COLUMNS},
           "SK_ID_PREV": "int32", "SK_ID_CURR": "int32"},
    keep_default_na=False,
    na_values=[""]
)

In [5]:
prev.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,FLAG_LAST_APPL_PER_CONTRACT,NAME_CONTRACT_STATUS,...,NAME_PAYMENT_TYPE,NAME_GOODS_CATEGORY,CHANNEL_TYPE,CNT_PAYMENT,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,Y,Approved,...,Cash through the bank,Mobile,Country-wide,12.0,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,Y,Approved,...,XNA,XNA,Contact center,36.0,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,Y,Approved,...,Cash through the bank,XNA,Credit and cash offices,12.0,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,Y,Approved,...,Cash through the bank,XNA,Credit and cash offices,12.0,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,Y,Refused,...,Cash through the bank,XNA,Credit and cash offices,24.0,Cash Street: high,NaN,NaN,NaN,NaN,NaN


In [6]:
prev.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 21 columns):
 #   Column                       Non-Null Count    Dtype   
---  ------                       --------------    -----   
 0   SK_ID_PREV                   1670214 non-null  int32   
 1   SK_ID_CURR                   1670214 non-null  int32   
 2   NAME_CONTRACT_TYPE           1670214 non-null  category
 3   AMT_ANNUITY                  1297979 non-null  float64 
 4   AMT_APPLICATION              1670214 non-null  float64 
 5   AMT_CREDIT                   1670213 non-null  float64 
 6   AMT_DOWN_PAYMENT             774370 non-null   float64 
 7   AMT_GOODS_PRICE              1284699 non-null  float64 
 8   FLAG_LAST_APPL_PER_CONTRACT  1670214 non-null  category
 9   NAME_CONTRACT_STATUS         1670214 non-null  category
 10  DAYS_DECISION                1670214 non-null  int64   
 11  NAME_PAYMENT_TYPE            1670214 non-null  category
 12  NAME_GOODS_CATEGORY          1670214 no

In [7]:
source_shape = (len(prev), len(source_columns))
loaded_shape = prev.shape
n_applicants = prev["SK_ID_CURR"].nunique()

print("Source dimensions:", source_shape)
print("Loaded dimensions:", loaded_shape)
print("Unique applications:", prev["SK_ID_PREV"].nunique())
print("Unique applicants:", n_applicants)
print("Memory used (MiB):", round(prev.memory_usage(deep=True).sum() / 1024**2, 1))

Source dimensions: (1670214, 37)
Loaded dimensions: (1670214, 21)
Unique applications: 1670214
Unique applicants: 338857
Memory used (MiB): 176.8


In [8]:
duplicate_prev_ids = prev["SK_ID_PREV"].duplicated().sum()
print("Duplicate application IDs:", duplicate_prev_ids)

assert prev[["SK_ID_PREV", "SK_ID_CURR"]].notna().all().all()
assert duplicate_prev_ids == 0, "Investigate repeated application IDs before aggregation."
assert "TARGET" not in prev.columns
assert n_applicants > 0
validation_results = {"Source application grain and nonmissing IDs": "PASS"}

Duplicate application IDs: 0


## 3. Understand the Values Before Aggregation

The EDA gives us a few important decisions:

- Approved, Refused, Canceled, and Unused offer are four different outcomes.
- Missing down payments are common for cash and revolving products; missing is not zero.
- Revolving loans have zero or missing `CNT_PAYMENT`, rather than a fixed installment term.
- `365243` is a placeholder in date fields, not an ordinary number of days.
- The amount fields of refused applications describe recorded requests/evaluations, not disbursed debt.

We check these patterns again instead of relying only on saved EDA outputs.

In [9]:
source_status_counts = prev["NAME_CONTRACT_STATUS"].value_counts(dropna=False)
source_status_counts

NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64

In [10]:
financial_inspection = pd.DataFrame({
    "missing": prev[AMOUNT_COLUMNS + ["CNT_PAYMENT"]].isna().sum(),
    "zero": prev[AMOUNT_COLUMNS + ["CNT_PAYMENT"]].eq(0).sum(),
    "negative": prev[AMOUNT_COLUMNS + ["CNT_PAYMENT"]].lt(0).sum(),
    "minimum": prev[AMOUNT_COLUMNS + ["CNT_PAYMENT"]].min(),
    "mean": prev[AMOUNT_COLUMNS + ["CNT_PAYMENT"]].mean(),
    "maximum": prev[AMOUNT_COLUMNS + ["CNT_PAYMENT"]].max()
})
financial_inspection

,missing,zero,negative,minimum,mean,maximum
AMT_APPLICATION,0,392402,0,0.0,175233.860360,6905160.000
AMT_CREDIT,1,336768,0,0.0,196114.021218,6905160.000
AMT_ANNUITY,372235,1637,0,0.0,15955.120659,418058.145
AMT_DOWN_PAYMENT,895844,369854,2,-0.9,6697.402139,3060045.000
AMT_GOODS_PRICE,385515,6869,0,0.0,227847.279283,6905160.000
CNT_PAYMENT,372230,144985,0,0.0,16.054082,84.000


In [11]:
term_state = pd.Series("Positive", index=prev.index)
term_state.loc[prev["CNT_PAYMENT"].isna()] = "Missing"
term_state.loc[prev["CNT_PAYMENT"].eq(0)] = "Zero"
term_state.loc[prev["CNT_PAYMENT"].lt(0)] = "Negative"
display(pd.crosstab(prev["NAME_CONTRACT_TYPE"], term_state))
display(pd.crosstab(prev["NAME_CONTRACT_TYPE"], prev["AMT_DOWN_PAYMENT"].isna()))
del term_state

col_0,Missing,Positive,Zero
NAME_CONTRACT_TYPE,,,
Cash loans,285990,461563,0
Consumer loans,37715,691436,0
Revolving loans,48179,0,144985
XNA,346,0,0


AMT_DOWN_PAYMENT,False,True
NAME_CONTRACT_TYPE,,
Cash loans,47934,699619
Consumer loans,724041,5110
Revolving loans,2395,190769
XNA,0,346


In [12]:
UNKNOWN_CODES = ["XNA", "XAP"]
category_inspection = pd.DataFrame({
    "missing": prev[CATEGORY_COLUMNS].isna().sum(),
    "XNA": prev[CATEGORY_COLUMNS].eq("XNA").sum(),
    "XAP": prev[CATEGORY_COLUMNS].eq("XAP").sum(),
    "distinct_observed": prev[CATEGORY_COLUMNS].nunique()
})
category_inspection

,missing,XNA,XAP,distinct_observed
NAME_CONTRACT_TYPE,0,346,0,4
NAME_CONTRACT_STATUS,0,0,0,4
NAME_PAYMENT_TYPE,0,627384,0,4
NAME_GOODS_CATEGORY,0,950809,0,28
CHANNEL_TYPE,0,0,0,8
PRODUCT_COMBINATION,346,0,0,17
FLAG_LAST_APPL_PER_CONTRACT,0,0,0,2


In [13]:
SENTINEL = 365243
date_inspection = pd.DataFrame({
    "missing": prev[DATE_COLUMNS].isna().sum(),
    "sentinel_365243": prev[DATE_COLUMNS].eq(SENTINEL).sum(),
    "minimum_without_sentinel": prev[DATE_COLUMNS].mask(prev[DATE_COLUMNS].eq(SENTINEL)).min(),
    "maximum_without_sentinel": prev[DATE_COLUMNS].mask(prev[DATE_COLUMNS].eq(SENTINEL)).max()
})
display(date_inspection)

assert prev["DAYS_DECISION"].notna().all()
assert prev["DAYS_DECISION"].lt(0).all(), "Previous decisions must precede the current application."

,missing,sentinel_365243,minimum_without_sentinel,maximum_without_sentinel
DAYS_DECISION,0,0,-2922.0,-1.0
DAYS_FIRST_DRAWING,673065,934444,-2922.0,-2.0
DAYS_FIRST_DUE,673065,40645,-2892.0,-2.0
DAYS_LAST_DUE_1ST_VERSION,673065,93864,-2801.0,2389.0
DAYS_LAST_DUE,673065,211221,-2889.0,-2.0
DAYS_TERMINATION,673065,225913,-2874.0,-2.0


## 4. Group by Applicant ID

An applicant can have several previous applications. Every feature group below
uses `SK_ID_CURR`, so it will already have one row per applicant before we combine it.

هنجمع على رقم العميل، مش رقم الطلب السابق.

In [14]:
prev_grouped = prev.groupby("SK_ID_CURR", sort=True)

print("Applicants:", prev_grouped.ngroups)
print("Previous applications:", len(prev))

Applicants: 338857
Previous applications: 1670214


## 5. Application Counts and Status

These features describe how often the applicant applied and what happened to
those applications. We first create readable row-level flags, then sum them.
This avoids running a Python function separately for hundreds of thousands of groups.

| Column | English explanation | المعنى بالعربي |
|---|---|---|
| `prev_total_applications` | Number of all previous application records. | إجمالي الطلبات السابقة. |
| `prev_approved_applications` | Number marked Approved. | عدد الطلبات المقبولة. |
| `prev_refused_applications` | Number marked Refused. | عدد الطلبات المرفوضة. |
| `prev_canceled_applications` | Number marked Canceled. | عدد الطلبات الملغاة. |
| `prev_unused_offer_applications` | Number marked Unused offer. | عدد العروض التي لم تُستخدم. |

In [15]:
STATUS_FEATURES = {
    "Approved": "prev_approved_applications",
    "Refused": "prev_refused_applications",
    "Canceled": "prev_canceled_applications",
    "Unused offer": "prev_unused_offer_applications"
}
assert prev["NAME_CONTRACT_STATUS"].isin(STATUS_FEATURES).all()

prev["is_approved"] = prev["NAME_CONTRACT_STATUS"].eq("Approved")
prev["is_refused"] = prev["NAME_CONTRACT_STATUS"].eq("Refused")
prev["is_canceled"] = prev["NAME_CONTRACT_STATUS"].eq("Canceled")
prev["is_unused_offer"] = prev["NAME_CONTRACT_STATUS"].eq("Unused offer")

In [16]:
prev_status_features = prev_grouped.agg(
    prev_total_applications=("SK_ID_PREV", "size"),
    prev_approved_applications=("is_approved", "sum"),
    prev_refused_applications=("is_refused", "sum"),
    prev_canceled_applications=("is_canceled", "sum"),
    prev_unused_offer_applications=("is_unused_offer", "sum")
).reset_index()

In [17]:
# All rates use total applications, including canceled and unused offers.
denominator = prev_status_features["prev_total_applications"].replace(0, np.nan)
prev_status_features["prev_approval_rate"] = prev_status_features["prev_approved_applications"] / denominator
prev_status_features["prev_refusal_rate"] = prev_status_features["prev_refused_applications"] / denominator
prev_status_features["prev_cancellation_rate"] = prev_status_features["prev_canceled_applications"] / denominator
prev_status_features["prev_unused_offer_rate"] = prev_status_features["prev_unused_offer_applications"] / denominator
RATE_COLUMNS = ["prev_approval_rate", "prev_refusal_rate", "prev_cancellation_rate", "prev_unused_offer_rate"]

The four rates measure each outcome's share of **all** previous applications.
A refusal rate of `0.25` means one quarter of the recorded applications were refused.

النسبة = عدد الطلبات بالحالة دي ÷ إجمالي الطلبات السابقة، مش المقبول والمرفوض بس.

In [18]:
# Independent per-applicant status counts from a cross-tabulation.
expected_status_counts = pd.crosstab(prev["SK_ID_CURR"], prev["NAME_CONTRACT_STATUS"])
expected_status_counts = expected_status_counts.reindex(columns=STATUS_FEATURES, fill_value=0).rename(columns=STATUS_FEATURES)
pd.testing.assert_frame_equal(
    prev_status_features.set_index("SK_ID_CURR")[list(STATUS_FEATURES.values())],
    expected_status_counts,
    check_dtype=False, check_names=False
)
assert prev_status_features[list(STATUS_FEATURES.values())].sum(axis=1).eq(prev_status_features["prev_total_applications"]).all()
assert prev_status_features["prev_total_applications"].sum() == len(prev)
assert prev_status_features[RATE_COLUMNS].notna().all().all()
assert prev_status_features[RATE_COLUMNS].ge(0).all().all()
assert prev_status_features[RATE_COLUMNS].le(1).all().all()
assert np.allclose(prev_status_features[RATE_COLUMNS].sum(axis=1), 1)
validation_results["Application status counts and rates"] = "PASS"
del expected_status_counts

In [19]:
prev_status_features.head()

,SK_ID_CURR,prev_total_applications,prev_approved_applications,prev_refused_applications,prev_canceled_applications,prev_unused_offer_applications,prev_approval_rate,prev_refusal_rate,prev_cancellation_rate,prev_unused_offer_rate
0,100001,1,1,0,0,0,1.0,0.0,0.0,0.0
1,100002,1,1,0,0,0,1.0,0.0,0.0,0.0
2,100003,3,3,0,0,0,1.0,0.0,0.0,0.0
3,100004,1,1,0,0,0,1.0,0.0,0.0,0.0
4,100005,2,1,0,1,0,0.5,0.0,0.5,0.0


## 6. Previous Application Amounts

We summarize the **recorded amounts across all application statuses**. These are
not outstanding debt or proof that the money was disbursed. Zeros stay as recorded;
for example, many canceled applications have zero requested amounts.

Means and maxima describe typical and largest values. The minimum requested
amount shows the lower end of recorded requests, including zero. Requested,
credit, and down-payment totals describe accumulated recorded application amounts.
We avoid summing annuities because applications may not represent simultaneous loans.
Goods-price mean/max give product-size information without another closely related total.

The inspection found **two negative down payments**. We retain the raw values and
count these anomalies, but exclude them from down-payment statistics. Other observed
nonnegative down payments, including zero, remain valid.

مفيش تعويض للقيم الناقصة بصفر. ومتوسط القيم بيتحسب من المعلومات المتاحة فقط.

In [20]:
prev["negative_down_payment"] = prev["AMT_DOWN_PAYMENT"].lt(0)
prev["down_payment_for_stats"] = prev["AMT_DOWN_PAYMENT"].where(prev["AMT_DOWN_PAYMENT"].ge(0))

print("Negative down payments excluded from statistics:", prev["negative_down_payment"].sum())

Negative down payments excluded from statistics: 2


In [21]:
prev_amount_features = prev_grouped.agg(
    prev_application_amount_mean=("AMT_APPLICATION", "mean"),
    prev_application_amount_max=("AMT_APPLICATION", "max"),
    prev_application_amount_min=("AMT_APPLICATION", "min"),
    prev_credit_amount_mean=("AMT_CREDIT", "mean"),
    prev_credit_amount_max=("AMT_CREDIT", "max"),
    prev_annuity_mean=("AMT_ANNUITY", "mean"),
    prev_annuity_max=("AMT_ANNUITY", "max"),
    prev_down_payment_mean=("down_payment_for_stats", "mean"),
    prev_down_payment_max=("down_payment_for_stats", "max"),
    prev_goods_price_mean=("AMT_GOODS_PRICE", "mean"),
    prev_goods_price_max=("AMT_GOODS_PRICE", "max")
)

In [22]:
# min_count=1 keeps an all-missing applicant total missing, rather than zero.
amount_totals = prev_grouped[["AMT_APPLICATION", "AMT_CREDIT", "down_payment_for_stats"]].sum(min_count=1)
amount_totals = amount_totals.rename(columns={
    "AMT_APPLICATION": "prev_application_amount_total",
    "AMT_CREDIT": "prev_credit_amount_total",
    "down_payment_for_stats": "prev_down_payment_total"
})
prev_amount_features = prev_amount_features.join(amount_totals, validate="one_to_one").reset_index()
del amount_totals

| Column family | English explanation | المعنى بالعربي |
|---|---|---|
| `prev_application_amount_mean/max/min/total` | Requested amount statistics, preserving recorded zero requests. | متوسط وأكبر وأقل وإجمالي المبلغ المطلوب. |
| `prev_credit_amount_mean/max/total` | Recorded credit/evaluation amount statistics. | ملخص مبالغ الائتمان المسجلة، وليس الدين الحالي. |
| `prev_annuity_mean/max` | Typical and highest available annuity. | متوسط وأكبر قسط مسجل. |
| `prev_down_payment_mean/max/total` | Available nonnegative down-payment statistics. | ملخص الدفعات المقدمة المتاحة وغير السالبة. |
| `prev_goods_price_mean/max` | Typical and largest available goods price. | متوسط وأكبر سعر للسلع. |

If an applicant has no usable value for a field, its statistics stay missing.
We keep availability counts later to help interpret these missing values.

In [23]:
amount_feature_definitions = {
    "prev_application_amount_mean": ("AMT_APPLICATION", "mean"),
    "prev_application_amount_max": ("AMT_APPLICATION", "max"),
    "prev_application_amount_min": ("AMT_APPLICATION", "min"),
    "prev_application_amount_total": ("AMT_APPLICATION", "sum"),
    "prev_credit_amount_mean": ("AMT_CREDIT", "mean"),
    "prev_credit_amount_max": ("AMT_CREDIT", "max"),
    "prev_credit_amount_total": ("AMT_CREDIT", "sum"),
    "prev_annuity_mean": ("AMT_ANNUITY", "mean"),
    "prev_annuity_max": ("AMT_ANNUITY", "max"),
    "prev_down_payment_mean": ("down_payment_for_stats", "mean"),
    "prev_down_payment_max": ("down_payment_for_stats", "max"),
    "prev_down_payment_total": ("down_payment_for_stats", "sum"),
    "prev_goods_price_mean": ("AMT_GOODS_PRICE", "mean"),
    "prev_goods_price_max": ("AMT_GOODS_PRICE", "max")
}
# Check null semantics for every applicant and every amount feature.
available_amount_counts = prev_grouped[AMOUNT_COLUMNS + ["down_payment_for_stats"]].count()
for feature, (source, statistic) in amount_feature_definitions.items():
    assert prev_amount_features.set_index("SK_ID_CURR")[feature].isna().eq(available_amount_counts[source].eq(0)).all()

In [24]:
prev_amount_features.drop(columns="SK_ID_CURR").describe()

,prev_application_amount_mean,prev_application_amount_max,prev_application_amount_min,prev_credit_amount_mean,prev_credit_amount_max,prev_annuity_mean,prev_annuity_max,prev_down_payment_mean,prev_down_payment_max,prev_goods_price_mean,prev_goods_price_max,prev_application_amount_total,prev_credit_amount_total,prev_down_payment_total
count,3.388570e+05,3.388570e+05,3.388570e+05,3.388570e+05,3.388570e+05,338377.000000,338377.000000,3.187530e+05,3.187530e+05,3.377930e+05,3.377930e+05,3.388570e+05,3.388570e+05,3.187530e+05
mean,1.544683e+05,3.775473e+05,4.405501e+04,1.703318e+05,4.167555e+05,14656.028970,23944.281570,7.118137e+03,1.176714e+04,1.953843e+05,3.787647e+05,8.637214e+05,9.666384e+05,1.627049e+04
std,1.526145e+05,4.219383e+05,9.350608e+04,1.649810e+05,4.574522e+05,10093.856491,19264.040276,1.827759e+04,2.873953e+04,1.924749e+05,4.220768e+05,1.353907e+06,1.495396e+06,4.228138e+04
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,6.259770e+04,9.957150e+04,0.000000e+00,6.741675e+04,1.061505e+05,7908.423750,10525.725000,0.000000e+00,0.000000e+00,7.460550e+04,1.005165e+05,1.503450e+05,1.595655e+05,0.000000e+00
50%,1.068345e+05,2.105100e+05,1.755000e+04,1.181655e+05,2.250000e+05,12092.197500,18081.270000,3.355071e+03,5.850000e+03,1.303740e+05,2.125080e+05,3.902040e+05,4.372785e+05,7.033500e+03
75%,1.916936e+05,4.635000e+05,5.393250e+04,2.164125e+05,5.539500e+05,18489.159000,31840.875000,8.325000e+03,1.350000e+04,2.475000e+05,4.635000e+05,1.028205e+06,1.178910e+06,1.889932e+04
max,4.050000e+06,6.905160e+06,4.050000e+06,4.050000e+06,6.905160e+06,300425.445000,418058.145000,2.025000e+06,3.060045e+06,4.050000e+06,6.905160e+06,3.827475e+07,4.359984e+07,6.441543e+06


## 7. Loan Terms and Planned Duration

We use positive `CNT_PAYMENT` values from Cash loans and Consumer loans to
describe proposed **fixed installment terms**. Refused applications can still have
proposed terms; these averages are not limited to granted loans.

Revolving loans have no fixed installment duration here. Their missing/zero terms
are counted separately instead of pulling installment averages towards zero.

For duration, we use the original scheduled span:

`DAYS_LAST_DUE_1ST_VERSION - DAYS_FIRST_DUE`

Only approved installment applications with usable dates and terms contribute.
Both dates must be non-sentinel, the first due date cannot precede the decision,
and the span cannot be negative. A one-payment schedule may have a valid span of zero.
This is the span between scheduled payments, not the full time from approval to closure.

The original last due date may be in the future because it is a **planned schedule**.
We do not use later realized closure dates as durations or future outcome labels.

In [25]:
prev["is_installment"] = prev["NAME_CONTRACT_TYPE"].isin(["Cash loans", "Consumer loans"])
prev["is_revolving"] = prev["NAME_CONTRACT_TYPE"].eq("Revolving loans")
prev["valid_term"] = prev["is_installment"] & prev["CNT_PAYMENT"].gt(0)
prev["cnt_payment_for_stats"] = prev["CNT_PAYMENT"].where(prev["valid_term"])

# Clean derived date columns only; the raw date columns remain available for audits.
prev["first_due_for_stats"] = prev["DAYS_FIRST_DUE"].mask(prev["DAYS_FIRST_DUE"].eq(SENTINEL))
prev["original_last_due_for_stats"] = prev["DAYS_LAST_DUE_1ST_VERSION"].mask(prev["DAYS_LAST_DUE_1ST_VERSION"].eq(SENTINEL))

In [26]:
planned_span = prev["original_last_due_for_stats"] - prev["first_due_for_stats"]
dates_available = prev[["first_due_for_stats", "original_last_due_for_stats"]].notna().all(axis=1)
invalid_schedule = dates_available & (
    planned_span.lt(0) | prev["first_due_for_stats"].lt(prev["DAYS_DECISION"])
)
prev["valid_planned_duration"] = (
    prev["is_approved"] & prev["valid_term"] & dates_available & ~invalid_schedule
)
prev["planned_duration_days"] = planned_span.where(prev["valid_planned_duration"])

print("Applications with valid fixed terms:", prev["valid_term"].sum())
print("Applications with valid planned duration:", prev["valid_planned_duration"].sum())
print("Date-order anomalies excluded from duration:", invalid_schedule.sum())
del planned_span, dates_available

Applications with valid fixed terms: 1152999
Applications with valid planned duration: 896021
Date-order anomalies excluded from duration: 0


In [27]:
prev_term_features = prev_grouped.agg(
    prev_cnt_payment_mean=("cnt_payment_for_stats", "mean"),
    prev_cnt_payment_max=("cnt_payment_for_stats", "max"),
    prev_valid_term_applications=("valid_term", "sum"),
    prev_planned_duration_days_mean=("planned_duration_days", "mean"),
    prev_planned_duration_days_max=("planned_duration_days", "max"),
    prev_valid_planned_duration_applications=("valid_planned_duration", "sum")
).reset_index()

| Column | English explanation | المعنى بالعربي |
|---|---|---|
| `prev_cnt_payment_mean/max` | Average and longest available fixed installment term, in payments. | متوسط وأكبر عدد أقساط للطلبات ذات المدة المحددة. |
| `prev_valid_term_applications` | Applications with a usable fixed installment count. | عدد الطلبات التي لها عدد أقساط صالح. |
| `prev_planned_duration_days_mean/max` | Average and longest usable original scheduled payment span, in days. | متوسط وأكبر مدة بين أول وآخر قسط في الجدول الأصلي. |
| `prev_valid_planned_duration_applications` | Approved installment applications contributing to scheduled duration. | عدد الطلبات المقبولة التي لها مدة مجدولة صالحة. |

In [28]:
assert prev_term_features["prev_valid_term_applications"].sum() == prev["valid_term"].sum()
assert prev_term_features["prev_valid_planned_duration_applications"].sum() == prev["valid_planned_duration"].sum()
assert prev.loc[prev["is_revolving"], "cnt_payment_for_stats"].isna().all()
assert prev_term_features["prev_cnt_payment_mean"].dropna().gt(0).all()
assert prev_term_features["prev_planned_duration_days_mean"].dropna().ge(0).all()
assert prev_term_features["prev_valid_planned_duration_applications"].le(prev_term_features["prev_valid_term_applications"]).all()
for feature in ["prev_cnt_payment_mean", "prev_cnt_payment_max"]:
    assert prev_term_features[feature].isna().eq(prev_term_features["prev_valid_term_applications"].eq(0)).all()
for feature in ["prev_planned_duration_days_mean", "prev_planned_duration_days_max"]:
    assert prev_term_features[feature].isna().eq(prev_term_features["prev_valid_planned_duration_applications"].eq(0)).all()
validation_results["Fixed-term and planned-duration eligibility"] = "PASS"

In [29]:
prev_term_features.head()

,SK_ID_CURR,prev_cnt_payment_mean,prev_cnt_payment_max,prev_valid_term_applications,prev_planned_duration_days_mean,prev_planned_duration_days_max,prev_valid_planned_duration_applications
0,100001,8.0,8.0,1,210.0,210.0,1
1,100002,24.0,24.0,1,690.0,690.0,1
2,100003,10.0,12.0,3,270.0,330.0,3
3,100004,4.0,4.0,1,90.0,90.0,1
4,100005,12.0,12.0,1,330.0,330.0,1


## 8. Application Timing

Dates use negative days relative to the current application. `-10` is more recent
than `-500`, so the **maximum** is the latest decision and the minimum is the oldest.
The EDA's final candidate-feature list accidentally suggests the minimum for recency;
we follow the actual date convention and the Bureau notebook's min/max pattern.

| Column | English explanation | المعنى بالعربي |
|---|---|---|
| `prev_days_decision_max` | Most recent decision, in negative relative days. | تاريخ أحدث قرار؛ القيمة الأقرب للصفر. |
| `prev_days_decision_min` | Oldest decision, in negative relative days. | تاريخ أقدم قرار؛ القيمة الأكثر سالبية. |
| `prev_days_since_decision_mean` | Average elapsed days since previous decisions, positive. | متوسط عدد الأيام التي مرت منذ القرارات السابقة. |

We keep only the positive mean elapsed time, rather than adding its identical
negative counterpart as another feature.

In [30]:
prev_timing_features = prev_grouped.agg(
    prev_days_decision_max=("DAYS_DECISION", "max"),
    prev_days_decision_min=("DAYS_DECISION", "min")
)
prev_timing_features["prev_days_since_decision_mean"] = -prev_grouped["DAYS_DECISION"].mean()
prev_timing_features = prev_timing_features.reset_index()

In [31]:
assert prev_timing_features["prev_days_decision_min"].le(prev_timing_features["prev_days_decision_max"]).all()
assert prev_timing_features["prev_days_decision_max"].lt(0).all()
assert prev_timing_features["prev_days_since_decision_mean"].gt(0).all()
assert prev_timing_features["prev_days_since_decision_mean"].ge(-prev_timing_features["prev_days_decision_max"]).all()
assert prev_timing_features["prev_days_since_decision_mean"].le(-prev_timing_features["prev_days_decision_min"]).all()

In [32]:
prev_timing_features.head()

,SK_ID_CURR,prev_days_decision_max,prev_days_decision_min,prev_days_since_decision_mean
0,100001,-1740,-1740,1740.0
1,100002,-606,-606,606.0
2,100003,-746,-2341,1305.0
3,100004,-815,-815,815.0
4,100005,-315,-757,536.0


## 9. Application Patterns

We count diversity in a small set of interpretable fields: loan type, product
combination, goods category, channel, and payment type. This describes the breadth
of previous interactions without creating a dummy feature for every category.

`XNA`, `XAP`, and genuine missing values are excluded from **known-category diversity**.
The original categorical values remain unchanged; unknown loan/payment-type counts
are kept in the availability group. Zero diversity means no known category was observed.

We also count positive down payments and nonfinal contract offers.
`FLAG_LAST_APPL_PER_CONTRACT = N` identifies an offer that was not the final contract
application. It is a record of offer revision, **not proof of repayment rescheduling**.
Differences between actual and original last due dates can reflect early repayment
or later changes; we do not label those differences as rescheduled offers.

In [33]:
PATTERN_COLUMNS = {
    "NAME_CONTRACT_TYPE": "contract_type_known",
    "PRODUCT_COMBINATION": "product_combination_known",
    "NAME_GOODS_CATEGORY": "goods_category_known",
    "CHANNEL_TYPE": "channel_type_known",
    "NAME_PAYMENT_TYPE": "payment_type_known"
}
for source, clean in PATTERN_COLUMNS.items():
    prev[clean] = prev[source].where(~prev[source].isin(UNKNOWN_CODES))

prev["has_down_payment"] = prev["AMT_DOWN_PAYMENT"].gt(0)
prev["nonfinal_contract_offer"] = prev["FLAG_LAST_APPL_PER_CONTRACT"].eq("N")
assert prev["FLAG_LAST_APPL_PER_CONTRACT"].isin(["Y", "N"]).all()

In [34]:
prev_pattern_features = prev_grouped.agg(
    prev_distinct_contract_types=("contract_type_known", "nunique"),
    prev_distinct_product_combinations=("product_combination_known", "nunique"),
    prev_distinct_goods_categories=("goods_category_known", "nunique"),
    prev_distinct_channels=("channel_type_known", "nunique"),
    prev_distinct_payment_types=("payment_type_known", "nunique"),
    prev_applications_with_down_payment=("has_down_payment", "sum"),
    prev_nonfinal_contract_applications=("nonfinal_contract_offer", "sum")
).reset_index()

| Column family | English explanation | المعنى بالعربي |
|---|---|---|
| `prev_distinct_contract_types/product_combinations/goods_categories/channels/payment_types` | Number of different known categories used by the applicant. | عدد الأنواع أو المنتجات أو القنوات المختلفة المعروفة للعميل. |
| `prev_applications_with_down_payment` | Applications with a strictly positive recorded down payment. | عدد الطلبات التي لها دفعة مقدمة أكبر من صفر. |
| `prev_nonfinal_contract_applications` | Applications flagged as nonfinal offers for the contract. | عدد العروض التي لم تكن الطلب النهائي للعقد. |

In [35]:
pattern_counts = prev_pattern_features.set_index("SK_ID_CURR")
assert pattern_counts.ge(0).all().all()
assert pattern_counts.le(prev_grouped.size(), axis=0).all().all()
assert prev_pattern_features["prev_applications_with_down_payment"].sum() == prev["AMT_DOWN_PAYMENT"].gt(0).sum()
assert prev_pattern_features["prev_nonfinal_contract_applications"].sum() == prev["FLAG_LAST_APPL_PER_CONTRACT"].eq("N").sum()
validation_results["Application pattern counts"] = "PASS"
del pattern_counts

In [36]:
prev_pattern_features.head()

,SK_ID_CURR,prev_distinct_contract_types,prev_distinct_product_combinations,prev_distinct_goods_categories,prev_distinct_channels,prev_distinct_payment_types,prev_applications_with_down_payment,prev_nonfinal_contract_applications
0,100001,1,1,1,1,1,1,0
1,100002,1,1,1,1,0,0,0
2,100003,2,3,2,3,1,1,0
3,100004,1,1,1,1,1,1,0
4,100005,2,2,1,2,1,1,0


## 10. Missingness and Data Availability

We count missing information without converting it into zero amounts or zero
durations. Each count below has a clear context:

| Column family | English explanation | المعنى بالعربي |
|---|---|---|
| `prev_missing_annuity_applications` | Applications with no annuity value; often no loan was issued. | عدد الطلبات التي لا تتوفر لها قيمة القسط. |
| `prev_missing_down_payment_applications` | All applications without a down-payment value. | إجمالي الطلبات التي لا تتوفر لها دفعة مقدمة. |
| `prev_consumer_missing_down_payment_applications` | Missing down payments in Consumer loans, where the field is usually available. | نقص الدفعة المقدمة داخل القروض الاستهلاكية. |
| `prev_cash_or_revolving_missing_down_payment_applications` | Missing down payments in product types where structural absence is common. | نقص الدفعة في القروض النقدية أو الدوارة؛ غالبًا حسب نوع المنتج. |
| `prev_zero_down_payment_applications` | Explicitly recorded zero down payments. | عدد الطلبات التي سجلت دفعة مقدمة تساوي صفرًا. |
| `prev_negative_down_payment_applications` | Negative recorded values excluded from amount statistics. | عدد القيم السالبة التي تحتاج مراجعة. |
| `prev_missing_cnt_payment_applications` | Missing installment counts across all applications. | إجمالي الطلبات التي لا يتوفر لها عدد أقساط. |
| `prev_installment_missing_term_applications` | Missing counts specifically for Cash/Consumer installment products. | نقص عدد الأقساط في المنتجات ذات المدة المحددة. |
| `prev_revolving_without_fixed_term_applications` | Revolving records with zero or missing fixed terms. | طلبات الائتمان الدوار التي ليس لها عدد أقساط ثابت. |
| `prev_unknown_contract_type_applications` / `prev_unknown_payment_type_applications` | Missing or placeholder categories, separate from known diversity. | عدد الطلبات ذات النوع غير المعروف أو غير المطبق. |
| `prev_missing_goods_price_applications` | No goods price, which may reflect product type. | عدد الطلبات التي لا يتوفر لها سعر سلع. |
| `prev_first_due_missing_applications` / `prev_original_last_due_missing_applications` | Genuinely empty scheduled date fields. | عدد حقول المواعيد المجدولة الناقصة فعليًا. |
| `prev_first_due_sentinel_applications` / `prev_original_last_due_sentinel_applications` | Date fields explicitly equal to `365243`, separate from empty fields. | عدد المواعيد التي تحتوي على القيمة الخاصة 365243. |

Product-specific missingness is context, not a definitive diagnosis. For example,
an empty Consumer-loan down payment can also belong to a refused application.
We do not automatically call every missing cash/revolving value structural.
Likewise, `365243` is recorded as a sentinel count rather than a confirmed active-loan flag.

In [37]:
prev["missing_annuity"] = prev["AMT_ANNUITY"].isna()
prev["missing_down_payment"] = prev["AMT_DOWN_PAYMENT"].isna()
prev["consumer_missing_down_payment"] = prev["NAME_CONTRACT_TYPE"].eq("Consumer loans") & prev["missing_down_payment"]
prev["cash_or_revolving_missing_down_payment"] = prev["NAME_CONTRACT_TYPE"].isin(["Cash loans", "Revolving loans"]) & prev["missing_down_payment"]
prev["zero_down_payment"] = prev["AMT_DOWN_PAYMENT"].eq(0)
prev["missing_cnt_payment"] = prev["CNT_PAYMENT"].isna()
prev["installment_missing_term"] = prev["is_installment"] & prev["missing_cnt_payment"]
prev["revolving_without_fixed_term"] = prev["is_revolving"] & (prev["CNT_PAYMENT"].isna() | prev["CNT_PAYMENT"].eq(0))
prev["unknown_contract_type"] = prev["NAME_CONTRACT_TYPE"].isna() | prev["NAME_CONTRACT_TYPE"].isin(UNKNOWN_CODES)
prev["unknown_payment_type"] = prev["NAME_PAYMENT_TYPE"].isna() | prev["NAME_PAYMENT_TYPE"].isin(UNKNOWN_CODES)
prev["missing_goods_price"] = prev["AMT_GOODS_PRICE"].isna()
prev["first_due_missing"] = prev["DAYS_FIRST_DUE"].isna()
prev["original_last_due_missing"] = prev["DAYS_LAST_DUE_1ST_VERSION"].isna()
prev["first_due_sentinel"] = prev["DAYS_FIRST_DUE"].eq(SENTINEL)
prev["original_last_due_sentinel"] = prev["DAYS_LAST_DUE_1ST_VERSION"].eq(SENTINEL)

In [38]:
prev_availability_features = prev_grouped.agg(
    prev_missing_annuity_applications=("missing_annuity", "sum"),
    prev_missing_down_payment_applications=("missing_down_payment", "sum"),
    prev_consumer_missing_down_payment_applications=("consumer_missing_down_payment", "sum"),
    prev_cash_or_revolving_missing_down_payment_applications=("cash_or_revolving_missing_down_payment", "sum"),
    prev_zero_down_payment_applications=("zero_down_payment", "sum"),
    prev_negative_down_payment_applications=("negative_down_payment", "sum"),
    prev_missing_cnt_payment_applications=("missing_cnt_payment", "sum"),
    prev_installment_missing_term_applications=("installment_missing_term", "sum"),
    prev_revolving_without_fixed_term_applications=("revolving_without_fixed_term", "sum"),
    prev_unknown_contract_type_applications=("unknown_contract_type", "sum"),
    prev_unknown_payment_type_applications=("unknown_payment_type", "sum"),
    prev_missing_goods_price_applications=("missing_goods_price", "sum"),
    prev_first_due_missing_applications=("first_due_missing", "sum"),
    prev_original_last_due_missing_applications=("original_last_due_missing", "sum"),
    prev_first_due_sentinel_applications=("first_due_sentinel", "sum"),
    prev_original_last_due_sentinel_applications=("original_last_due_sentinel", "sum")
).reset_index()

In [39]:
availability = prev_availability_features.set_index("SK_ID_CURR")
assert availability.ge(0).all().all()
assert availability.le(prev_grouped.size(), axis=0).all().all()
assert not (prev["first_due_missing"] & prev["first_due_sentinel"]).any()
assert not (prev["original_last_due_missing"] & prev["original_last_due_sentinel"]).any()
# Missing, observed zero, positive, and negative down payments are distinct states.
assert (prev["missing_down_payment"].astype("int8") + prev["zero_down_payment"].astype("int8")
        + prev["has_down_payment"].astype("int8") + prev["negative_down_payment"].astype("int8")).eq(1).all()
assert availability["prev_missing_down_payment_applications"].sum() == prev["AMT_DOWN_PAYMENT"].isna().sum()
assert availability["prev_missing_cnt_payment_applications"].sum() == prev["CNT_PAYMENT"].isna().sum()
validation_results["Separate missing, zero, negative, and sentinel states"] = "PASS"
del availability

In [40]:
prev_availability_features.head()

,SK_ID_CURR,prev_missing_annuity_applications,prev_missing_down_payment_applications,prev_consumer_missing_down_payment_applications,prev_cash_or_revolving_missing_down_payment_applications,prev_zero_down_payment_applications,prev_negative_down_payment_applications,prev_missing_cnt_payment_applications,prev_installment_missing_term_applications,prev_revolving_without_fixed_term_applications,prev_unknown_contract_type_applications,prev_unknown_payment_type_applications,prev_missing_goods_price_applications,prev_first_due_missing_applications,prev_original_last_due_missing_applications,prev_first_due_sentinel_applications,prev_original_last_due_sentinel_applications
0,100001,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,100002,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0
2,100003,0,1,0,1,1,0,0,0,0,0,1,0,0,0,0,0
3,100004,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,100005,1,1,0,1,0,0,1,1,0,0,1,1,1,1,0,0


## 11. Combine the Applicant-Level Feature Groups

Each table already has one row per applicant. We merge these summaries using
`validate="one_to_one"`, as in the Bureau notebook. No raw application rows are
merged into the final applicant table.

In [41]:
feature_groups = {
    "Application counts and status": prev_status_features,
    "Recorded amounts": prev_amount_features,
    "Fixed terms and planned duration": prev_term_features,
    "Application timing": prev_timing_features,
    "Application patterns": prev_pattern_features,
    "Missingness and availability": prev_availability_features
}
for name, feature_group in feature_groups.items():
    assert feature_group["SK_ID_CURR"].is_unique, name
    assert len(feature_group) == n_applicants, name

previous_application_features = prev_status_features.copy()
for feature_group in list(feature_groups.values())[1:]:
    previous_application_features = previous_application_features.merge(
        feature_group, on="SK_ID_CURR", how="left", validate="one_to_one"
    )
previous_application_features = previous_application_features.sort_values("SK_ID_CURR").reset_index(drop=True)

In [42]:
print("Shape:", previous_application_features.shape)
print("Unique applicants:", previous_application_features["SK_ID_CURR"].nunique())
previous_application_features.head()

Shape: (338857, 56)
Unique applicants: 338857


,SK_ID_CURR,prev_total_applications,prev_approved_applications,prev_refused_applications,prev_canceled_applications,prev_unused_offer_applications,prev_approval_rate,prev_refusal_rate,prev_cancellation_rate,prev_unused_offer_rate,...,prev_missing_cnt_payment_applications,prev_installment_missing_term_applications,prev_revolving_without_fixed_term_applications,prev_unknown_contract_type_applications,prev_unknown_payment_type_applications,prev_missing_goods_price_applications,prev_first_due_missing_applications,prev_original_last_due_missing_applications,prev_first_due_sentinel_applications,prev_original_last_due_sentinel_applications
0,100001,1,1,0,0,0,1.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,100002,1,1,0,0,0,1.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0
2,100003,3,3,0,0,0,1.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0
3,100004,1,1,0,0,0,1.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,100005,2,1,0,1,0,0.5,0.0,0.5,0.0,...,1,1,0,0,1,1,1,1,0,0


In [43]:
previous_application_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 338857 entries, 0 to 338856
Data columns (total 56 columns):
 #   Column                                                    Non-Null Count   Dtype  
---  ------                                                    --------------   -----  
 0   SK_ID_CURR                                                338857 non-null  int32  
 1   prev_total_applications                                   338857 non-null  int64  
 2   prev_approved_applications                                338857 non-null  int64  
 3   prev_refused_applications                                 338857 non-null  int64  
 4   prev_canceled_applications                                338857 non-null  int64  
 5   prev_unused_offer_applications                            338857 non-null  int64  
 6   prev_approval_rate                                        338857 non-null  float64
 7   prev_refusal_rate                                         338857 non-null  float64
 8   prev_cancellati

## 12. Validation

We check the final grain, exact source applicant coverage, status totals, and
rate ranges. For amounts, we compare a reproducible sample of applicants directly
with their source records and check total conservation across the entire dataset.
For timing, we independently sort decisions to verify every applicant's endpoints.

Finally, we inspect missing feature values and verify that they correspond to
unavailable inputs rather than accidental merge gaps.

In [44]:
assert previous_application_features["SK_ID_CURR"].notna().all()
assert previous_application_features["SK_ID_CURR"].is_unique
assert previous_application_features["SK_ID_CURR"].duplicated().sum() == 0
assert len(previous_application_features) == n_applicants
assert np.array_equal(previous_application_features["SK_ID_CURR"].to_numpy(), np.sort(prev["SK_ID_CURR"].unique()))
assert previous_application_features.columns.is_unique
assert previous_application_features["prev_total_applications"].gt(0).all()
for status, feature in STATUS_FEATURES.items():
    assert previous_application_features[feature].sum() == source_status_counts[status]
for feature, count in zip(RATE_COLUMNS, STATUS_FEATURES.values()):
    assert previous_application_features[feature].between(0, 1).all()
    np.testing.assert_allclose(previous_application_features[feature], previous_application_features[count] / previous_application_features["prev_total_applications"])
validation_results["Final applicant IDs, coverage, status totals, and rate formulas"] = "PASS"

In [45]:
# Include random applicants and boundary cases with missing/zero/negative inputs.
check_ids = previous_application_features["SK_ID_CURR"].sample(20, random_state=42).tolist()
for mask in [prev["negative_down_payment"], prev["AMT_DOWN_PAYMENT"].isna(),
             prev["AMT_DOWN_PAYMENT"].eq(0), prev["is_revolving"]]:
    check_ids += prev.loc[mask, "SK_ID_CURR"].head(2).tolist()
check_ids += prev_term_features.loc[prev_term_features["prev_valid_term_applications"].eq(0), "SK_ID_CURR"].head(2).tolist()
check_rows = prev.loc[prev["SK_ID_CURR"].isin(check_ids)]
check_features = previous_application_features.set_index("SK_ID_CURR").loc[sorted(set(check_ids))]

In [46]:
for applicant_id, records in check_rows.groupby("SK_ID_CURR"):
    actual = check_features.loc[applicant_id]
    for feature, (source, statistic) in amount_feature_definitions.items():
        values = records[source].dropna().to_numpy()
        # An all-missing sum should also be missing.
        expected = getattr(values, statistic)() if len(values) else np.nan
        np.testing.assert_allclose(actual[feature], expected, rtol=1e-12, equal_nan=True)
    np.testing.assert_allclose(actual["prev_days_since_decision_mean"], -records["DAYS_DECISION"].mean())
    for source, mean_feature, max_feature in [
        ("cnt_payment_for_stats", "prev_cnt_payment_mean", "prev_cnt_payment_max"),
        ("planned_duration_days", "prev_planned_duration_days_mean", "prev_planned_duration_days_max")
    ]:
        values = records[source].dropna().to_numpy()
        np.testing.assert_allclose(actual[mean_feature], values.mean() if len(values) else np.nan, equal_nan=True)
        np.testing.assert_allclose(actual[max_feature], values.max() if len(values) else np.nan, equal_nan=True)

for source, feature in [("AMT_APPLICATION", "prev_application_amount_total"),
                        ("AMT_CREDIT", "prev_credit_amount_total"),
                        ("down_payment_for_stats", "prev_down_payment_total")]:
    np.testing.assert_allclose(previous_application_features[feature].sum(min_count=1), prev[source].sum(min_count=1), rtol=1e-12)
print("Applicants checked directly against source records:", len(check_features))
validation_results["Direct source amount/term checks and population total conservation"] = "PASS"
del check_rows, check_features

Applicants checked directly against source records: 30


In [47]:
chronological_decisions = prev[["SK_ID_CURR", "DAYS_DECISION"]].sort_values("DAYS_DECISION")
latest_decisions = chronological_decisions.drop_duplicates("SK_ID_CURR", keep="last").set_index("SK_ID_CURR")["DAYS_DECISION"].sort_index()
oldest_decisions = chronological_decisions.drop_duplicates("SK_ID_CURR", keep="first").set_index("SK_ID_CURR")["DAYS_DECISION"].sort_index()
np.testing.assert_array_equal(previous_application_features["prev_days_decision_max"], latest_decisions)
np.testing.assert_array_equal(previous_application_features["prev_days_decision_min"], oldest_decisions)
validation_results["Every applicant's latest and oldest decision from sorted source records"] = "PASS"
del chronological_decisions, latest_decisions, oldest_decisions

### Inspect Missing Values

Missing amount statistics mean there was no usable observed value for that applicant.
Missing fixed-term statistics mean no eligible installment term was available.
Missing planned durations mean no approved installment application had a usable
original schedule. Counts, status rates, timing, and diversity should be complete.

الصفر في العدّ معناه إن مفيش سجل من النوع ده. أما القيمة الناقصة في المتوسط أو
المدة فمعناها إن المعلومات اللازمة للحساب مش متاحة.

In [48]:
missing_summary = pd.DataFrame({
    "missing_applicants": previous_application_features.isna().sum(),
    "missing_percent": (previous_application_features.isna().mean() * 100).round(2)
}).sort_values("missing_applicants", ascending=False)
display(missing_summary.loc[missing_summary["missing_applicants"].gt(0)])

nullable_features = set(amount_feature_definitions) | {
    "prev_cnt_payment_mean", "prev_cnt_payment_max",
    "prev_planned_duration_days_mean", "prev_planned_duration_days_max"
}
complete_columns = [col for col in previous_application_features if col not in nullable_features]
assert previous_application_features[complete_columns].notna().all().all()
for feature, (source, statistic) in amount_feature_definitions.items():
    assert previous_application_features.set_index("SK_ID_CURR")[feature].isna().eq(available_amount_counts[source].eq(0)).all()
validation_results["Feature missingness matches data availability"] = "PASS"

,missing_applicants,missing_percent
prev_down_payment_total,20104,5.93
prev_down_payment_mean,20104,5.93
prev_down_payment_max,20104,5.93
prev_planned_duration_days_mean,4764,1.41
prev_planned_duration_days_max,4764,1.41
prev_cnt_payment_max,2696,0.80
prev_cnt_payment_mean,2696,0.80
prev_goods_price_mean,1064,0.31
prev_goods_price_max,1064,0.31
prev_annuity_mean,480,0.14


In [49]:
numeric_features = previous_application_features.select_dtypes(include="number")
assert not np.isinf(numeric_features.to_numpy()).any()
validation_results["No unexpected infinities"] = "PASS"

with PREV_PATH.open("rb") as raw_file:
    raw_sha256_after = hashlib.file_digest(raw_file, "sha256").hexdigest()
assert raw_sha256_before == raw_sha256_after
validation_results["Raw CSV SHA-256 unchanged"] = "PASS"
display(pd.Series(validation_results, name="Result").to_frame())
del numeric_features

,Result
Source application grain and nonmissing IDs,PASS
Application status counts and rates,PASS
Fixed-term and planned-duration eligibility,PASS
Application pattern counts,PASS
"Separate missing, zero, negative, and sentinel states",PASS
"Final applicant IDs, coverage, status totals, and rate formulas",PASS
Direct source amount/term checks and population total conservation,PASS
Every applicant's latest and oldest decision from sorted source records,PASS
Feature missingness matches data availability,PASS
No unexpected infinities,PASS


## 13. Save the Applicant Features

We use the same interim-data convention as `bureau_features.parquet`.
The output contains `SK_ID_CURR` plus the engineered numeric features; no index is saved.

In [50]:
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
previous_application_features.to_parquet(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)

Saved: /run/media/momen/Life/Projects/DEPI Final Project/data/interim/previous_application_features.parquet


## 14. Read the Saved File Back

Loading successfully is only the first check. We also compare shape, applicant IDs,
column types, values, and missingness with the feature table we just created.

In [51]:
saved_previous_application = pd.read_parquet(OUTPUT_PATH)

print("Saved shape:", saved_previous_application.shape)
print("Unique IDs:", saved_previous_application["SK_ID_CURR"].is_unique)
print("Duplicate IDs:", saved_previous_application["SK_ID_CURR"].duplicated().sum())

Saved shape: (338857, 56)
Unique IDs: True
Duplicate IDs: 0


In [52]:
assert saved_previous_application.shape == previous_application_features.shape
assert saved_previous_application["SK_ID_CURR"].is_unique
assert saved_previous_application["SK_ID_CURR"].duplicated().sum() == 0
pd.testing.assert_frame_equal(saved_previous_application, previous_application_features, check_exact=True)
validation_results["Parquet reload exactly preserves shape, IDs, dtypes, values, and nulls"] = "PASS"

In [53]:
saved_previous_application.head()

,SK_ID_CURR,prev_total_applications,prev_approved_applications,prev_refused_applications,prev_canceled_applications,prev_unused_offer_applications,prev_approval_rate,prev_refusal_rate,prev_cancellation_rate,prev_unused_offer_rate,...,prev_missing_cnt_payment_applications,prev_installment_missing_term_applications,prev_revolving_without_fixed_term_applications,prev_unknown_contract_type_applications,prev_unknown_payment_type_applications,prev_missing_goods_price_applications,prev_first_due_missing_applications,prev_original_last_due_missing_applications,prev_first_due_sentinel_applications,prev_original_last_due_sentinel_applications
0,100001,1,1,0,0,0,1.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,100002,1,1,0,0,0,1.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0
2,100003,3,3,0,0,0,1.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0
3,100004,1,1,0,0,0,1.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,100005,2,1,0,1,0,0.5,0.0,0.5,0.0,...,1,1,0,0,1,1,1,1,0,0


## 15. Final Summary

We now have one feature row per applicant with previous application history.
The summary below is calculated from this execution, including the saved output.

In [54]:
n_engineered_features = previous_application_features.shape[1] - 1
category_summary = "\n".join(
    f"| {name} | {group.shape[1] - 1} |" for name, group in feature_groups.items()
)
validation_summary = "\n".join(
    f"| {check} | {result} |" for check, result in validation_results.items()
)
display(Markdown(f"""
| Summary | Result |
|---|---|
| Source dataset dimensions | {source_shape[0]:,} rows × {source_shape[1]} columns |
| Selected columns loaded | {loaded_shape[1]} |
| Unique applicants | {n_applicants:,} |
| Engineered features, excluding SK_ID_CURR | {n_engineered_features} |
| Final and reloaded shape | {saved_previous_application.shape} |
| Output file | `{OUTPUT_PATH.relative_to(REPO_ROOT)}` |
| Raw file unchanged | SHA-256 matched |
| Negative down-payment records retained and flagged | {int(prev['negative_down_payment'].sum()):,} |
| Scheduled date-order anomalies excluded from duration | {int(invalid_schedule.sum()):,} |

| Feature category | Features |
|---|---|
{category_summary}

| Validation | Result |
|---|---|
{validation_summary}
"""))


| Summary | Result |
|---|---|
| Source dataset dimensions | 1,670,214 rows × 37 columns |
| Selected columns loaded | 21 |
| Unique applicants | 338,857 |
| Engineered features, excluding SK_ID_CURR | 55 |
| Final and reloaded shape | (338857, 56) |
| Output file | `data/interim/previous_application_features.parquet` |
| Raw file unchanged | SHA-256 matched |
| Negative down-payment records retained and flagged | 2 |
| Scheduled date-order anomalies excluded from duration | 0 |

| Feature category | Features |
|---|---|
| Application counts and status | 9 |
| Recorded amounts | 14 |
| Fixed terms and planned duration | 6 |
| Application timing | 3 |
| Application patterns | 7 |
| Missingness and availability | 16 |

| Validation | Result |
|---|---|
| Source application grain and nonmissing IDs | PASS |
| Application status counts and rates | PASS |
| Fixed-term and planned-duration eligibility | PASS |
| Application pattern counts | PASS |
| Separate missing, zero, negative, and sentinel states | PASS |
| Final applicant IDs, coverage, status totals, and rate formulas | PASS |
| Direct source amount/term checks and population total conservation | PASS |
| Every applicant's latest and oldest decision from sorted source records | PASS |
| Feature missingness matches data availability | PASS |
| No unexpected infinities | PASS |
| Raw CSV SHA-256 unchanged | PASS |
| Parquet reload exactly preserves shape, IDs, dtypes, values, and nulls | PASS |


### Decisions and Limitations

- **Amounts:** All application statuses contribute to recorded amount summaries.
  Totals do not measure outstanding debt or actual disbursement. Observed zeros
  remain recorded zeros; all-missing statistics stay missing. Two negative down
  payments are flagged and excluded only from down-payment statistics.
- **Terms and duration:** Revolving products are separated from fixed installment
  terms. Duration uses eligible approved applications' original planned payment
  span, not actual closure or a duration inferred from installment count.
- **Offer revisions:** Nonfinal contract offers are counted using the explicit flag.
  They should not be interpreted as confirmed repayment rescheduling.
- **Availability:** Product-context counts distinguish common structural absence
  from missing fields in installment/consumer products without claiming the exact
  reason for each missing value. `XNA`/`XAP` are preserved in the working source
  categories and excluded from known diversity. Date sentinels are counted separately
  from truly empty dates, without assuming they always mean an active/open-ended loan.
- **Historical scope:** Every decision is before the current application date.
  Planned future due dates can be known at that earlier decision; actual future
  outcomes are not used. Production use still requires confirming the source was
  available as of scoring time. No target, train/test linkage, or target-derived
  transformation is used. Applicants absent from this source are handled later,
  when aggregated feature tables are joined to the main applicant table.

No raw records were removed. Eligibility masks affect only the statistics that
need them. The existing Bureau notebook and its output were not changed.